# 🎬 Geração de Vídeo com Prompt + Imagem (via AnimateDiff)

In [ ]:
# ✅ Instalação e login Hugging Face!pip install diffusers transformers accelerate einops ffmpeg-python gradio --quiet!git clone https://github.com/guoyww/AnimateDiff.git!pip install -r AnimateDiff/requirements.txtfrom huggingface_hub import loginlogin()  # Vai pedir seu token Hugging Face (copie de https://huggingface.co/settings/tokens)

In [ ]:
import osimport torchimport ffmpegfrom PIL import Imageimport gradio as grfrom diffusers import AnimateDiffPipeline, MotionAdapterfrom diffusers.utils import export_to_videoos.makedirs("inputs", exist_ok=True)os.makedirs("outputs", exist_ok=True)

In [ ]:
def generate_video(image_path, prompt, duration=2, fps=14):    try:        print(f"🎥 Gerando vídeo para: {prompt}")        adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)        pipe = AnimateDiffPipeline.from_pretrained(            "SG161222/Realistic_Vision_V5.1_noVAE",            motion_adapter=adapter,            torch_dtype=torch.float16        )        pipe.enable_model_cpu_offload()        pipe.enable_vae_slicing()        pipe.to("cuda")        image = Image.open(image_path).convert("RGB")        image = image.resize((512, 512))        result = pipe(            prompt=prompt,            image=image,            num_frames=int(duration * fps),            guidance_scale=7.5,            num_inference_steps=25        )        frames = result.frames[0]        frame_dir = "outputs/temp_frames"        os.makedirs(frame_dir, exist_ok=True)        for idx, frame in enumerate(frames):            frame.save(f"{frame_dir}/frame_{idx:03d}.png")        output_path = os.path.join("outputs", os.path.basename(image_path).split(".")[0] + "_out.mp4")        (            ffmpeg            .input(f"{frame_dir}/frame_%03d.png", framerate=fps)            .output(output_path, vcodec='libx264', pix_fmt='yuv420p')            .run(overwrite_output=True)        )        print("✅ Vídeo gerado:", output_path)        return output_path    except Exception as e:        print("❌ Erro ao gerar vídeo:", str(e))        return None

In [ ]:
def video_app(img, prompt, duration, fps):    result = generate_video(img, prompt, duration, fps)    if result is None:        raise gr.Error("❌ Falha na geração. Veja o console para detalhes.")    return result

In [ ]:
with gr.Blocks() as demo:    gr.Markdown("## 🌅 Vídeo Cinematográfico: Prompt + Imagem com AnimateDiff")    with gr.Row():        img_input = gr.Image(label="Imagem de Entrada", type="filepath")        prompt_input = gr.Textbox(label="Prompt (ex: 'waves moving, sunset, sand blowing')")    with gr.Row():        dur_input = gr.Slider(label="Duração (s)", minimum=1, maximum=10, step=0.5, value=2)        fps_input = gr.Slider(label="FPS", minimum=6, maximum=30, step=1, value=14)    gen_btn = gr.Button("🎬 Gerar Vídeo")    output_video = gr.Video(label="🎞️ Resultado")    gen_btn.click(fn=video_app, inputs=[img_input, prompt_input, dur_input, fps_input], outputs=output_video)    gr.Markdown("🔑 Se ainda não fez login: execute `huggingface-cli login` no topo e cole seu token Hugging Face.")    demo.launch(share=True, debug=True)